# 20. Final submission

Group 22: Jalil-g, rdevasia25, Gtpatel02. Issue #39. One standalone notebook: raw CSVs in, `submission.csv` out.

## Goal

Train the final model on all 10000 labeled rows and write P(AI) for the 3000 test rows. This is the model behind Kaggle submission 58222131 (public AUROC 0.98553).

**Final model**

```
P(AI) = sigmoid(0.75 * classical + 0.25 * ELECTRA)
classical = 0.5 * style + 0.5 * text
```

- **Style:** HistGradientBoosting on 147 style features (lengths, vocab richness, case, digits, punctuation, blank lines, function words). No word identity. Trained on originals plus a whitespace-collapsed copy at weight .25 so it does not lean only on blank lines.
- **Text:** mean of word 1-3 + case char 2-5 TF-IDF logistic regression (C 16) and a case char 2-5 linear SVM (C 1) trained on rows with no blank line. Plus 2 when the text has a blank line, only if no human training row has one.
- **ELECTRA:** `google/electra-small-discriminator` fine-tuned locally, 3 epochs, 256 tokens. Its tokenizer ignores whitespace, so it covers the formatting weak spot.
- Every score is standardized on the 3000 unlabeled rows before mixing. No test-batch normalization.

Where the choices came from: style #13/#14, text #10/#11, style+text weights #17, ELECTRA #16, ELECTRA weight .25 picked on training rows in #19. All picks were made before this notebook, inside training rows.

## Setup

Runs in `envs/electra` (torch 2.15.0, transformers 5.22.2). One sklearn thread. Seed 42. Data in `data/raw/`.

In [1]:
import hashlib, json, math, platform, re, string, subprocess, time, unicodedata, warnings
import importlib.metadata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from scipy.special import expit, logit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from threadpoolctl import threadpool_limits
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, OUT = ROOT / "data" / "raw", ROOT / "submissions" / "20_final_submission"
SEED = 42
DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

# Final blend.
NEURAL_WEIGHT = .25          # picked on inner training rows in #19
STYLE_WEIGHT, TEXT_WEIGHT = .5, .5   # picked on inner training rows in #17
# Style arm (#13 HistGB, #14 collapse copies).
HIST = {"max_iter": 250, "learning_rate": .06, "max_leaf_nodes": 15, "min_samples_leaf": 30,
        "l2_regularization": 5., "early_stopping": False, "random_state": SEED}
COPY_WEIGHT = .25
# Text arm (#10/#11).
WORD13 = {"analyzer": "word", "ngram_range": (1, 3), "min_df": 2, "max_features": None, "lowercase": True}
CASECHAR = {"analyzer": "char", "ngram_range": (2, 5), "min_df": 3, "max_features": 300000, "lowercase": False}
LR = {"C": 16., "solver": "liblinear", "max_iter": 5000, "random_state": SEED, "tol": 1e-4}
SVM = {"C": 1., "dual": "auto", "max_iter": 20000, "random_state": SEED, "tol": 1e-4}
BLANK_OFFSET = 2.
# ELECTRA (#16).
ELECTRA_REPO, ELECTRA_REVISION = "google/electra-small-discriminator", "fa8239aadc095e9164941d05878b98afe9b953c3"
MAX_TOKENS, BATCH, EVAL_BATCH, LEARNING_RATE, WEIGHT_DECAY, WARMUP_FRACTION, EPOCHS = 256, 16, 64, 3e-5, .01, .1, 3
START = time.perf_counter()
print("Device:", DEVICE, "| torch", torch.__version__)

/Users/jaliljabbarli/Cornell/Cornell-Text-Authorship/envs/electra/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps | torch 2.15.0


## Checks

Read IDs as text. Labels must be 0/1 and test IDs must match the sample file. The 3000 unlabeled rows are the score reference (the #1 audit found no duplicate groups, so all of them are eligible).

In [2]:
def require(ok, message):
    if not ok:
        raise ValueError(message)

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

read = lambda name: pd.read_csv(RAW / name, dtype={"id": "string"})
train, unlabeled, test, sample = read("train.csv"), read("unlabeled.csv"), read("test.csv"), read("sample_submission.csv")
for name, frame in [("train", train), ("unlabeled", unlabeled), ("test", test)]:
    require(frame.id.is_unique and frame.text.notna().all(), f"{name}: bad IDs or text")
require(len(train) == 10000 and set(train.label) == {0, 1} and train.label.mean() == .5, "Train must be 10000 balanced rows")
require(len(test) == 3000 and set(test.id) == set(sample.id) and list(sample.columns) == ["id", "label"], "Test and sample differ")
require(not set(test.id) & set(train.id), "Test overlaps train")
DATA_SHA256 = {n: file_hash(RAW / n) for n in ["train.csv", "unlabeled.csv", "test.csv", "sample_submission.csv"]}
print(f"Train {len(train)} (AI share {train.label.mean():.2f}), unlabeled reference {len(unlabeled)}, test {len(test)}")

Train 10000 (AI share 0.50), unlabeled reference 3000, test 3000


## Method

Three arms. Each one fits on training text only and is standardized with its own mean/std on the unlabeled rows.

In [3]:
# Style features (#13). 147 numbers per text, no word identity.
TOKEN, SENTENCE_SPLIT, PARAGRAPH_SPLIT = re.compile(r"\w+"), re.compile(r"[.!?]+"), re.compile(r"\n[ \t]*\n")
LIST_LINE, CONTRACTION = re.compile(r"^\s*(?:[-*\u2022]|\d+[.)])\s"), re.compile(r"\w['\u2019]\w")
URL, REPEATED_SPACE, HEADING_END = re.compile(r"https?://|www\.", re.IGNORECASE), re.compile(r" {2,}"), ".!?,;:"
PRONOUNS = frozenset("i me my mine myself we us our ours ourselves you your yours yourself yourselves".split())
FUNCTION_WORDS = ("the of and to a in is that it for was on with as be by at this are from "
                  "or an but not have has had which they their there been were would could can will "
                  "more also however such these those than then into about very just so").split()
PUNCT_MARKS = {unicodedata.name(c).lower().replace(" ", "_").replace("-", "_"): c for c in string.punctuation}
PUNCT_MARKS.update({"en_dash": "\u2013", "em_dash": "\u2014", "left_single_quote": "\u2018", "right_single_quote": "\u2019",
                    "left_double_quote": "\u201c", "right_double_quote": "\u201d", "ellipsis": "\u2026", "bullet": "\u2022"})

def summary(values):
    if not values:
        return [0.0] * 7
    v = np.asarray(values, dtype=float)
    return [v.mean(), v.std(), v.min(), v.max(), *np.quantile(v, [.25, .5, .75])]

def one_text(text):
    text = str(text)
    chars, words, tokens = len(text), text.split(), TOKEN.findall(text.lower())
    sentences = [s for s in SENTENCE_SPLIT.split(text) if s.strip()]
    lines = text.split("\n") if text else []
    filled = [line for line in lines if line.strip()]
    paragraphs = [p for p in PARAGRAPH_SPLIT.split(text) if p.strip()]
    counts, letters = Counter(tokens), sum(c.isalpha() for c in text)
    n_words, n_tokens, n_types = max(1, len(words)), max(1, len(tokens)), max(1, len(counts))
    n_chars, n_lines, n_sentences = max(1, chars), max(1, len(lines)), max(1, len(sentences))
    word_letters = [[c for c in w if c.isalpha()] for w in words]
    entropy = -sum(c / n_tokens * math.log2(c / n_tokens) for c in counts.values()) if tokens else 0.0
    row = [
        math.log1p(chars), math.log1p(len(words)), math.log1p(len(sentences)), math.log1p(len(lines)), math.log1p(len(paragraphs)),
        len(counts) / n_tokens, sum(c == 1 for c in counts.values()) / n_types, sum(c == 2 for c in counts.values()) / n_types, entropy,
        sum(c.isupper() for c in text) / max(1, letters), sum(c.isdigit() for c in text) / n_chars,
        text.count(" ") / n_chars, sum(ord(c) > 127 for c in text) / n_chars, text.count("\n") / n_chars,
        sum(not line.strip() for line in lines) / n_lines, text.count("\t") / n_chars,
        sum(not c.isalnum() and not c.isspace() for c in text) / n_chars,
        sum(len(w) >= 2 and all(c.isupper() for c in w) for w in word_letters) / n_words,
        sum(len(w) >= 1 and w[0].isupper() and all(c.islower() for c in w[1:]) for w in word_letters) / n_words,
        sum(len(w) >= 7 for w in word_letters) / n_words, sum(1 <= len(w) <= 3 for w in word_letters) / n_words,
        len(CONTRACTION.findall(text)) / n_words,
        sum(bool(LIST_LINE.match(line)) for line in lines) / n_lines,
        sum(len(line.split()) <= 8 and not line.strip().endswith(tuple(HEADING_END)) for line in filled) / n_lines,
        sum(line[:1] in (" ", "\t") for line in filled) / n_lines,
        len(REPEATED_SPACE.findall(text)) / n_words, len(URL.findall(text)) / n_words,
        sum(counts[p] for p in PRONOUNS) / n_tokens, sum(len(s.split()) < 4 for s in sentences) / n_sentences]
    for values in [[len(t) for t in tokens], [len(s.split()) for s in sentences],
                   [len(line.split()) for line in filled], [len(p.split()) for p in paragraphs]]:
        row += summary(values)
    row += [text.count(mark) for mark in PUNCT_MARKS.values()]
    row += [counts[word] / n_tokens for word in FUNCTION_WORDS]
    return row

def style_features(texts):
    X = np.asarray([one_text(t) for t in texts], dtype=np.float32)
    require(X.shape == (len(texts), 147) and np.isfinite(X).all(), "Bad style features")
    return X

def has_blank_line(texts):
    return np.asarray(["\n\n" in str(t) for t in texts], dtype=bool)

def ref_stats(values):
    values = np.asarray(values, dtype=float)
    require(np.isfinite(values).all() and values.std() > 1e-12, "Bad reference scores")
    return float(values.mean()), float(values.std())

def standardize(values, stats):
    return (np.asarray(values, dtype=float) - stats[0]) / stats[1]

class StyleArm:
    def fit(self, texts, labels, reference):
        texts, labels = list(texts), np.asarray(labels)
        collapsed = [" ".join(str(t).split()) for t in texts]  # whitespace-only copy, same words, same label
        X = np.vstack([style_features(texts), style_features(collapsed)])
        weights = np.r_[np.ones(len(texts)), np.full(len(texts), COPY_WEIGHT)]
        self.model = HistGradientBoostingClassifier(**HIST).fit(X, np.r_[labels, labels], sample_weight=weights)
        self.stats = ref_stats(self.raw(reference))
        return self
    def raw(self, texts):
        p = self.model.predict_proba(style_features(list(texts)))[:, list(self.model.classes_).index(1)]
        return logit(np.clip(p, 1e-7, 1 - 1e-7))
    def score(self, texts):
        return standardize(self.raw(texts), self.stats)

class TextArm:
    def fit(self, texts, labels, reference):
        texts, labels, reference = pd.Series(list(texts)), np.asarray(labels), pd.Series(list(reference))
        keep = ~has_blank_line(texts)
        # The blank-line rule only switches on if no human training text has a blank line.
        self.rule_active = not bool(has_blank_line(texts[labels == 0]).any())
        tfidf = lambda s: TfidfVectorizer(**s, sublinear_tf=True, norm="l2", dtype=np.float32)
        self.word = Pipeline([("features", FeatureUnion([("word", tfidf(WORD13)), ("char", tfidf(CASECHAR))])),
                              ("classifier", LogisticRegression(**LR))]).fit(texts, labels)
        self.specialist = Pipeline([("features", tfidf(CASECHAR)), ("classifier", LinearSVC(**SVM))]).fit(texts[keep], labels[keep])
        self.word_stats = ref_stats(self.word.decision_function(reference))
        self.specialist_stats = ref_stats(self.specialist.decision_function(reference[~has_blank_line(reference)]))
        self.stats = ref_stats(self.raw(reference))
        return self
    def raw(self, texts):
        texts = list(texts)
        mean = .5 * (standardize(self.word.decision_function(texts), self.word_stats)
                     + standardize(self.specialist.decision_function(texts), self.specialist_stats))
        return mean + BLANK_OFFSET * has_blank_line(texts) * self.rule_active
    def score(self, texts):
        return standardize(self.raw(texts), self.stats)

class ElectraArm:
    def fit(self, texts, labels, reference):
        torch.manual_seed(SEED)
        self.tokenizer = AutoTokenizer.from_pretrained(ELECTRA_REPO, revision=ELECTRA_REVISION)
        self.model = AutoModelForSequenceClassification.from_pretrained(ELECTRA_REPO, revision=ELECTRA_REVISION, num_labels=2).to(DEVICE)
        ids, y = self.encode(texts), torch.tensor(np.asarray(labels), dtype=torch.long)
        total = math.ceil(len(ids) / BATCH) * EPOCHS
        decay = [p for n, p in self.model.named_parameters() if not (n.endswith("bias") or "LayerNorm" in n)]
        no_decay = [p for n, p in self.model.named_parameters() if n.endswith("bias") or "LayerNorm" in n]
        optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.}], lr=LEARNING_RATE)
        scheduler = get_linear_schedule_with_warmup(optimizer, round(WARMUP_FRACTION * total), total)
        generator, self.history = torch.Generator().manual_seed(SEED), []
        for epoch in range(1, EPOCHS + 1):
            self.model.train(); order, losses, t = torch.randperm(len(ids), generator=generator).tolist(), [], time.perf_counter()
            for i in range(0, len(order), BATCH):
                idx = order[i:i + BATCH]
                loss = self.model(**self.batch([ids[j] for j in idx]), labels=y[idx].to(DEVICE)).loss
                loss.backward(); optimizer.step(); scheduler.step(); optimizer.zero_grad(); losses.append(loss.item())
            self.history.append({"epoch": epoch, "mean_train_loss": float(np.mean(losses)), "seconds": time.perf_counter() - t})
            print(self.history[-1])
        self.stats = ref_stats(self.raw(reference))
        return self
    def encode(self, texts):
        return self.tokenizer(list(map(str, texts)), truncation=True, max_length=MAX_TOKENS)["input_ids"]
    def batch(self, input_ids):
        return {k: v.to(DEVICE) for k, v in self.tokenizer.pad({"input_ids": input_ids}, return_tensors="pt").items()}
    @torch.no_grad()
    def raw(self, texts):
        self.model.eval(); ids, out = self.encode(texts), []
        for i in range(0, len(ids), EVAL_BATCH):
            logits = self.model(**self.batch(ids[i:i + EVAL_BATCH])).logits.float()
            out.append((logits[:, 1] - logits[:, 0]).cpu().numpy())
        return np.concatenate(out).astype(float)
    def score(self, texts):
        return standardize(self.raw(texts), self.stats)

def final_p_ai(arms, texts):
    texts = list(texts)
    with threadpool_limits(limits=1):
        classical = STYLE_WEIGHT * arms["style"].score(texts) + TEXT_WEIGHT * arms["text"].score(texts)
    classical = standardize(classical, arms["classical_stats"])
    return expit((1 - NEURAL_WEIGHT) * classical + NEURAL_WEIGHT * arms["electra"].score(texts))

## Run

Fit all three arms on the 10000 labeled rows, then score test. ELECTRA takes most of the time.

In [4]:
timings, arms = {}, {}
with threadpool_limits(limits=1), warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)  # an unfinished fit should fail, not pass quietly
    t = time.perf_counter(); arms["style"] = StyleArm().fit(train.text, train.label, unlabeled.text); timings["style_fit"] = time.perf_counter() - t
    t = time.perf_counter(); arms["text"] = TextArm().fit(train.text, train.label, unlabeled.text); timings["text_fit"] = time.perf_counter() - t
    arms["classical_stats"] = ref_stats(STYLE_WEIGHT * arms["style"].score(unlabeled.text) + TEXT_WEIGHT * arms["text"].score(unlabeled.text))
t = time.perf_counter(); arms["electra"] = ElectraArm().fit(train.text, train.label, unlabeled.text); timings["electra_fit"] = time.perf_counter() - t
t = time.perf_counter(); p_test = final_p_ai(arms, test.text); timings["test_scoring"] = time.perf_counter() - t
# Score again in shuffled order and match by ID, to catch row shifts.
order = np.random.default_rng(SEED).permutation(len(test))
shuffled = pd.Series(final_p_ai(arms, test.text.iloc[order]), index=test.id.iloc[order].to_numpy())
shuffle_diff = float(np.max(np.abs(p_test - shuffled.loc[test.id].to_numpy())))
timings["total"] = time.perf_counter() - START
print({k: round(v, 1) for k, v in timings.items()}, "| blank-line rule active:", arms["text"].rule_active, "| shuffle diff", f"{shuffle_diff:.2g}")

W1110 22:10:34.450000 15926 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 29217.86it/s]

{'epoch': 1, 'mean_train_loss': 0.32424169970154765, 'seconds': 119.24414895905647}


{'epoch': 2, 'mean_train_loss': 0.09549193086028099, 'seconds': 116.40817483293358}


{'epoch': 3, 'mean_train_loss': 0.048267509827762844, 'seconds': 117.4281672090292}


{'style_fit': 21.4, 'text_fit': 42.7, 'electra_fit': 363.4, 'test_scoring': 16.7, 'total': 470.1} | blank-line rule active: True | shuffle diff 0


## Results

No held-out score here: every label is used for training. The held-out evidence lives in #17/#19 (frozen 2000 with 8000-fit models) and on Kaggle. We check the file and compare it with the file we submitted. ELECTRA on Apple MPS is not bitwise repeatable and this notebook fits in plain `train.csv` order, so small differences are expected.

In [5]:
submission = sample[["id"]].merge(pd.DataFrame({"id": test.id, "label": p_test}), on="id", how="left", validate="one_to_one")
require(list(submission.columns) == ["id", "label"] and len(submission) == 3000 and submission.id.is_unique, "Bad submission shape")
require(submission.label.notna().all() and submission.label.between(0, 1).all(), "Labels must be finite in [0, 1]")
require(shuffle_diff <= 1e-6, "Scores depend on row order")
in_sample = float(roc_auc_score(train.label, final_p_ai(arms, train.text)))  # direction check only, not held out
require(in_sample > .9, "Class direction looks wrong")
checks = {"rows": len(submission), "min": float(submission.label.min()), "max": float(submission.label.max()),
          "share_above_0_5": float((submission.label >= .5).mean()), "in_sample_auroc_not_heldout": in_sample}
submitted_path = ROOT / "submissions/18_final_checks/20261111T002651559290Z/submission.csv"
if submitted_path.exists():  # Kaggle 58222131
    old = pd.read_csv(submitted_path, dtype={"id": "string"}).set_index("id").label.loc[submission.id]
    checks["vs_submitted"] = {"max_abs_diff": float(np.max(np.abs(old.to_numpy() - submission.label.to_numpy()))),
                              "spearman": float(old.rank().corr(pd.Series(submission.label.to_numpy(), index=old.index).rank()))}
display(pd.Series({k: v for k, v in checks.items() if k != "vs_submitted"}, name="submission").to_frame())
print("Vs submitted Kaggle file 58222131:", checks.get("vs_submitted", "file not found, skipped"))
display(pd.DataFrame(arms["electra"].history).round(4))

,submission
rows,3000.000000
min,0.176090
max,0.837552
share_above_0_5,0.453667
in_sample_auroc_not_heldout,1.000000


Vs submitted Kaggle file 58222131: {'max_abs_diff': 0.08730308985496227, 'spearman': 0.9968521485391275}


,epoch,mean_train_loss,seconds
0,1,0.3242,119.2441
1,2,0.0955,116.4082
2,3,0.0483,117.4282


## Takeaway

- Tried: the full final model (style + text + ELECTRA at .25) trained from raw CSVs on all 10000 rows in one notebook. 470 s on Apple MPS, ELECTRA 363 s of it.
- Found: the file passes every check (3000 unique IDs, labels .176 to .838, shuffle diff 0). Against submitted Kaggle file 58222131, Spearman .9969 and max diff .087. That is a retrain, not a copy: ELECTRA on MPS is not repeatable and this notebook fits in `train.csv` order, while #18 used split-manifest order for the text and ELECTRA fits.
- Next: keep 58222131 (public .98553) as the final Kaggle pick. This notebook is the clean recipe for the report. Its own file has no Kaggle score; uploading it would only measure retrain noise.

## Save

Write `submissions/20_final_submission/<run>/submission.csv` (ignored) and one safe record in `results/20_final_submission/`. Models are not saved. Nothing is uploaded; Jalil uploads by hand.

In [6]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
out = OUT / run_id; out.mkdir(parents=True, exist_ok=False)
submission.to_csv(out / "submission.csv", index=False)
record = {
    "status": "completed", "run_id": run_id, "utc": datetime.now(timezone.utc).isoformat(),
    "git_sha": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
    "command": "uv run --project envs/electra --locked jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.timeout=7200 notebooks/20_final_submission.ipynb",
    "seed": SEED, "fit_rows": len(train), "eval_rows": 0, "validation_auroc": None,
    "heldout_confirmation": {"kaggle_ref": 58222131, "kaggle_public_auroc": 0.98553, "weight_pick": "results/19_internal_neural_pick/20261111T024705598557Z.json"},
    "config": {"neural_weight": NEURAL_WEIGHT, "style_weight": STYLE_WEIGHT, "text_weight": TEXT_WEIGHT, "hist": HIST, "copy_weight": COPY_WEIGHT,
               "word13": {**WORD13, "ngram_range": list(WORD13["ngram_range"])}, "casechar": {**CASECHAR, "ngram_range": list(CASECHAR["ngram_range"])},
               "lr": LR, "svm": SVM, "blank_offset": BLANK_OFFSET, "electra": {"repo": ELECTRA_REPO, "revision": ELECTRA_REVISION,
               "max_tokens": MAX_TOKENS, "batch": BATCH, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "warmup_fraction": WARMUP_FRACTION, "epochs": EPOCHS},
               "reference": "all 3000 unlabeled rows", "fit_order": "train.csv order"},
    "data_sha256": DATA_SHA256, "lock_sha256": file_hash(ROOT / "envs/electra/uv.lock"),
    "versions": {n: importlib.metadata.version(n) for n in ["numpy", "pandas", "scipy", "scikit-learn", "torch", "transformers", "tokenizers"]}
                | {"python": platform.python_version()},
    "hardware": {"device": DEVICE, "platform": platform.platform()},
    "metrics": {**checks, "blank_rule_active": arms["text"].rule_active, "electra_history": arms["electra"].history, "shuffle_diff": shuffle_diff},
    "runtime_seconds": timings["total"], "timings": timings,
    "artifacts": {str((out / "submission.csv").relative_to(ROOT)): file_hash(out / "submission.csv")},
    "kaggle": {"uploaded_by_notebook": False},
}
(ROOT / "results" / "20_final_submission").mkdir(parents=True, exist_ok=True)
record_path = ROOT / "results" / "20_final_submission" / f"{run_id}.json"
record_path.write_text(json.dumps(record, indent=2, sort_keys=True) + "\n")
print("Wrote", (out / "submission.csv").relative_to(ROOT), "and", record_path.relative_to(ROOT))

Wrote submissions/20_final_submission/20261111T031804535719Z/submission.csv and results/20_final_submission/20261111T031804535719Z.json
